**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mills-2026"], check=True)
    os.chdir("/content/encore/entries/mills-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Mills, Lee, Howe, Li et al. (2026) on DANDI:001203

Paper: Mills et al., "Amygdalostriatal transition zone neurons encode sustained cue responses to guide
defensive behaviors", Neuron (2026), doi:10.1016/j.neuron.2026.08.012.
Claims: `claims/mills-2026.yaml`, written from the author-accepted manuscript (Europe PMC
PMC13592531) with the supplement's figure legends, and the paper's code at
github.com/Tyelab/Mills-et-al-2026, commit 1fee26d.

Dataset: [DANDI:001203](https://dandiarchive.org/dandiset/001203/0.261103.1719), version
0.261103.1719, doi:10.48324/dandi.001203/0.261103.1719. The paper cites the dandiset without a
version. This is the only published version (released 2026-11-03, "Initial release following
publication in Neuron"), and the draft has the same 706 assets, so it is pinned. Every file is
streamed from the archive; nothing is written to disk.

What the dandiset holds, as the inventory below shows: 13 Open Ephys sessions (12 mice, one
deposited twice) with spikes, trial events, port entries and SLEAP poses; 13 Neuropixels
recordings with spikes but no trial events, no behavior and no per-unit region; 34 calcium
imaging discrimination files (33 mice, one deposited twice) and 22 contextual fear imaging
sessions (11 mice), with CNMF-E traces, cue events and poses. There are no snRNA-seq, RNAscope,
optogenetic or unpredicted-US data.

| Claim | Tested here |
|---|---|
| fig2-recorded-neurons (link) | tested |
| figs9-sustained-only-cs-shock | tested, 10 of 15 paired ASt mice (Open Ephys) |
| figs10-no-reward-relationship | tested, the across-trials test only (paired mice) |
| fig2o-region-differs-cs-reward | tested, the ASt part only (no TS units) |
| fig2q-trajectories-diverge | tested, paired Open Ephys mice |
| figs8-single-trial-divergence | tested, paired part only |
| fig3h-cs-decoded-paired-only | tested, paired part only |
| figs13-imaging-mice-discriminate | tested |
| fig5j-drd2-cs-shock-responses | tested on all CNMF-E ROIs |
| fig5h-drd1-no-difference | tested on all CNMF-E ROIs |
| fig5k-drd2-cs-reward-no-difference | tested on all CNMF-E ROIs |
| fig5m-drd2-rich-clusters-cs-shock | tested on all CNMF-E ROIs |
| figs15-no-context-encoding | tested |
| figs5, figs10lm, fig3fg | blocked: pose of the Open Ephys sessions is not on the trial clock (a fixed offset is supported by two checks in 2 ASt mice only) |
| fig2e, fig2o-ast-greater, fig2p | blocked: no unit is labeled TS, and the Neuropixels files lack trial events; ASt against CeA is shown for the record |
| figs7 (four), figs10b, figs10-high-defensive, fig3h-behavior, fig3j to fig3m | blocked: no trial events or behavior for any unpaired ephys mouse |
| figs14, figs14q | blocked: no unpredicted-US imaging session in the dandiset |
| fig1l, figs4, fig3b, fig3c, fig4 (three), figs12gh, fig6 (three), figs16, figs17 | blocked: RNAscope and optogenetics not deposited |
| fig1b, fig1j, figs2 | blocked: the snRNA-seq data are at GEO GSE211437, outside the dandiset; the paper's pipeline (Seurat, Libra, edgeR in R) cannot be installed in the entry's locked Python environment |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, zlib, resource, collections
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import scipy.stats as ss
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.signal import find_peaks
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import mills as M
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
T0 = time.time()
KEY = "mills-2026"
def seed(claim, k=0):
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k
print("dandiset", M.DANDISET, "version", M.VERSION)

## Data check

Inventory of every NWB file: kind, subject, group and genotype as the files give them, units or
ROIs, and which tables are present. Then the contents loaded for the analyses, duplicates, and the
checks of the description against the values.

In [ ]:
nwb = M.nwb_assets()
def kind(p):
    if "ecephys+image" in p: return "openephys"
    if "_ecephys.nwb" in p: return "neuropixels"
    return "imaging"
def inv(a):
    f = M.open_nwb(a["path"]); p = a["path"]
    r = {"file": p.split("/")[-1][:58], "kind": kind(p), "GB": round(a["size"] / 1e9, 1)}
    sub = f["general/subject"]
    r["subject_description"] = M._s(sub["description"]) if "description" in sub else ""
    r["genotype"] = M._s(sub["genotype"]) if "genotype" in sub else ""
    r["session_description"] = M._s(f["session_description"])[:40]
    r["trials"] = "intervals/trials" in f
    r["pose"] = "processing/behavior/PoseEstimationTrack0" in f
    if r["kind"] != "imaging":
        loc = M._strs(f["general/extracellular_ephys/electrodes/location"])
        r["electrode_locations"] = ",".join(sorted(set(loc)))
        r["units"] = f["units/id"].shape[0]
        r["unit_columns_with_region"] = any(c in f["units"] for c in ("location", "brain_area", "electrodes"))
        r["units_below_50Hz"] = int((f["units/firing_rate"][:] < 50).sum()) if "firing_rate" in f["units"] else None
        if "label" in f["units"] and "firing_rate" in f["units"]:
            lab = M._strs(f["units/label"]); r["good_units_below_50Hz"] = int(((lab == "good") & (f["units/firing_rate"][:] < 50)).sum())
        if "probe_depth" in f["units"]:
            pdp = f["units/probe_depth"][:]; r["unit probe_depth um"] = f"{pdp.min():.0f}-{pdp.max():.0f}"
        r["histology images"] = "acquisition/NeuropixelsHistologyImages" in f
    else:
        r["rois"] = f["processing/ophys/ImageSegmentation/PlaneSegmentation/id"].shape[0] if "processing/ophys" in f else 0
        r["roi_columns"] = ",".join(c for c in f["processing/ophys/ImageSegmentation/PlaneSegmentation"] if c not in ("id", "image_mask", "reference_images")) if "processing/ophys" in f else ""
    r["path"] = p
    return r
with ThreadPoolExecutor(8) as ex:
    INV = pd.DataFrame(list(ex.map(inv, nwb)))
print(INV.groupby("kind").size())
print(INV.drop(columns=["path"]).to_string())

In [ ]:
# Open Ephys sessions: spikes, trial events, port entries, poses. The Neuropixels files hold no
# trial events and no behavior (inventory above), so no cue response can be computed from them.
oe_paths = INV.loc[INV.kind == "openephys", "path"].tolist()
with ThreadPoolExecutor(4) as ex:
    OE_ALL = list(ex.map(M.load_oe, oe_paths))
# one recording deposited twice: identical spikes and events
dup = [d for d in OE_ALL if d["subject_id"].startswith("6197")]
print("6197 files:", [d["path"].split("/")[-1] for d in dup])
print("identical spike trains:", all(np.array_equal(a, b) for a, b in zip(dup[0]["spikes"], dup[1]["spikes"])),
      "identical CS onsets:", np.array_equal(dup[0]["shock_on"], dup[1]["shock_on"]) and np.array_equal(dup[0]["reward_on"], dup[1]["reward_on"]))
OE, seen = [], set()
for d in sorted(OE_ALL, key=lambda d: d["path"]):
    if d["subject_id"] in seen: continue
    seen.add(d["subject_id"]); OE.append(d)
for d in OE:
    rate = np.array([len(s) / d["duration"] for s in d["spikes"]])
    d["rates"] = rate
    d["units"] = [s for s, r in zip(d["spikes"], rate) if r < 50]  # code: discall.py, rate below 50 Hz
    d["region"] = "ASt" if d["location"] == ["amygdalostriatal transition zone"] else "CeA"
    lat = []
    for o in d["reward_on"]:
        pe = d["port_on"][d["port_on"] >= o]; lat.append(pe[0] - o if len(pe) else np.inf)
    d["latency"] = np.array(lat)                        # reading: first port entry at or after onset
    d["latency_tone"] = d["reward_off"] - d["reward_on"]  # reading: tone duration (collection ends the tone)
AST = [d for d in OE if d["region"] == "ASt"]
tab = pd.DataFrame([{"subject": d["subject_id"], "region": d["region"], "units": len(d["spikes"]), "units<50Hz": len(d["units"]),
                     "max rate Hz": d["rates"].max().round(1), "CS-shock": len(d["shock_on"]), "CS-reward": len(d["reward_on"]),
                     "port entries": len(d["port_on"]), "session s": round(d["duration"]),
                     "shock tone s": np.round(np.median(d["shock_off"] - d["shock_on"]), 2),
                     "reward tone s (min-max)": f'{(d["reward_off"]-d["reward_on"]).min():.2f}-{(d["reward_off"]-d["reward_on"]).max():.2f}',
                     "pose frames": len(d["pose_t"]), "pose fps": round(1 / np.median(np.diff(d["pose_t"])), 2)} for d in OE])
print(tab.to_string())
print("ASt mice", len(AST), "ASt units below 50 Hz", sum(len(d["units"]) for d in AST))

In [ ]:
# Check: the reward tone ends when the reward is collected (STAR Methods). Tone stop minus first port entry after onset.
gap = np.concatenate([(d["reward_off"] - d["reward_on"]) - d["latency"] for d in OE])
gap = gap[np.isfinite(gap)]
print("tone stop minus first port entry, s: median %.3f, 5th-95th pct %.3f to %.3f, n=%d" % (np.median(gap), *np.percentile(gap, [5, 95]), len(gap)))
print(pd.DataFrame([{"subject": d["subject_id"][:12], "CS-reward": len(d["reward_on"]),
                     "in port at onset": int(sum(((d["port_on"] < o) & (d["port_off"] > o)).any() for o in d["reward_on"])),
                     "reward delivery s after onset (median)": np.median([d["reward_delivery"][d["reward_delivery"] >= o][0] - o for o in d["reward_on"] if (d["reward_delivery"] >= o).any()]),
                     "tone 20 s (not collected)": int((d["latency_tone"] > 19.9).sum()),
                     "latency>2 s, tone": int((d["latency_tone"] > 2).sum()), "latency>2 s, first entry": int((d["latency"] > 2).sum())} for d in OE]).to_string())
# Check: spikes share the clock of the trial table. Share of units with |mean z| > 2 over 0-1 s after the
# true CS onsets, and after the same onsets shifted 45 s back (inside the inter-trial interval).
rows = []
for d in OE:
    def share(on):
        out = []
        for s in d["units"]:
            z, c = M.cue_z(s, on, 0.1); out.append(abs(M.window_mean(z, c, (0, 1))) > 2)
        return np.mean(out)
    rows.append({"subject": d["subject_id"][:12], "shock true": share(d["shock_on"]), "shock -45 s": share(d["shock_on"] - 45),
                 "reward true": share(d["reward_on"]), "reward -45 s": share(d["reward_on"] - 45)})
print(pd.DataFrame(rows).round(2).to_string())

The pose of the Open Ephys sessions is checked against the trial clock in two independent ways:
the nose position at the port-entry times of the trial table (the nose is at the port then, so
its positions cluster only when the clocks agree), and the speed of the haunch around the shock
times (a 0.7 mA shock makes the mouse jump). Each is computed at lags from -300 to +300 s. The
imaging files, whose port entries were detected from the same behavior video, are the control.

In [ ]:
LAGS = np.arange(-300, 300.01, 0.2)
def port_lag(t, xy, pe):
    nose = xy[:, M.NODES.index("Nose")]
    spread = []
    for L in LAGS:
        i = np.clip(np.searchsorted(t, pe + L), 0, len(t) - 1)
        p = nose[i]; ok = ~np.isnan(p[:, 0])
        spread.append(np.median(np.linalg.norm(p[ok] - np.median(p[ok], 0), axis=1)))
    spread = np.array(spread); k = np.argmin(spread)
    return LAGS[k], spread[k], spread[np.argmin(abs(LAGS))], np.median(spread)
def shock_lag(d):
    t, xy = d["pose_t"], d["pose"]; ppc = M.pixels_per_cm(xy); fps = 1 / np.median(np.diff(t))
    sp = np.nan_to_num(np.linalg.norm(np.diff(xy[:, M.NODES.index("Haunch")], axis=0), axis=1) / ppc * fps)
    g = np.arange(0, t[-1], 0.1); s = np.interp(g, t[1:], sp)
    idx = np.clip(((d["shock_relay"][:, None] + LAGS[None, :]) / 0.1).astype(int), 0, len(s) - 1)
    eta = s[idx].mean(0); k = np.argmax(eta)
    return LAGS[k], (eta[k] - np.median(eta)) / eta.std()
rows = []
for d in OE:
    L, best, at0, med = port_lag(d["pose_t"], d["pose"], d["port_on"])
    Ls, zs = shock_lag(d)
    rows.append({"subject": d["subject_id"][:12], "pose fps": round(1 / np.median(np.diff(d["pose_t"]))), "port: best lag s": round(L, 1),
                 "nose spread px at best": round(best, 1), "at lag 0": round(at0, 1), "median over lags": round(med, 1),
                 "shock: best lag s": round(Ls, 1), "shock peak z": round(zs, 1),
                 "pose span s": round(d["pose_t"][-1]), "ephys span s": round(d["duration"])})
POSE_CLOCK = pd.DataFrame(rows)
print(POSE_CLOCK.to_string())

In [ ]:
# Offset and drift: the best port-entry lag in each quarter of the session (port entries split into
# four equal groups by time). A constant lag with a small spread in all four quarters is a fixed offset.
rows = []
for d in OE:
    pe = d["port_on"]; edges = np.quantile(pe, [0, 0.25, 0.5, 0.75, 1]); r = {"subject": d["subject_id"][:12]}
    for k, (a, b) in enumerate(zip(edges[:-1], edges[1:])):
        L, best, _, _ = port_lag(d["pose_t"], d["pose"], pe[(pe >= a) & (pe <= b)])
        r[f"Q{k+1} lag s"] = round(L, 1); r[f"Q{k+1} spread px"] = round(best, 1)
    rows.append(r)
POSE_DRIFT = pd.DataFrame(rows); print(POSE_DRIFT.to_string())

In [ ]:
# Calcium imaging sessions. The two J933 files hold identical traces; their port entries differ.
im_paths = INV.loc[(INV.kind == "imaging") & (INV.rois > 0), "path"].tolist()
with ThreadPoolExecutor(4) as ex:
    IM_ALL = list(ex.map(M.load_ophys, im_paths))
j = [d for d in IM_ALL if "J933" in d["path"]]
print("J933 files:", [(d["path"].split("/")[-1], d["F"].shape, len(d["port_on"])) for d in j],
      "identical traces:", np.array_equal(j[0]["F"], j[1]["F"]))
for d in IM_ALL:
    d["file"] = d["path"].split("/")[-1][:45]
    d["gen"] = "Drd2" if "drd2" in d["genotype"].lower() else "Drd1a"
    d["dt"] = float(np.median(np.diff(d["t"])))
    if "port_on" in d:
        d["port_lag"] = port_lag(d["pose_t"], d["pose"], d["port_on"])
print(pd.DataFrame([{"file": d["file"], "port lag s": round(d["port_lag"][0], 1), "spread at best px": round(d["port_lag"][1], 1),
                     "at 0": round(d["port_lag"][2], 1), "median": round(d["port_lag"][3], 1)} for d in IM_ALL if "port_lag" in d]).to_string())

In [ ]:
# Keep the J933 file whose port entries agree with its pose (lag near 0), drop the other.
bad = [d for d in IM_ALL if "J933" in d["path"] and abs(d["port_lag"][0]) > 5]
print("dropped:", [d["path"] for d in bad])
IM = [d for d in IM_ALL if d not in bad]
DISC = [d for d in IM if "cs_on" in d]
CTX = [d for d in IM if "Context" in d["session_description"]]
for d in DISC:
    d["grp"] = "paired" if "paired with" in d["session_description"] else "unpaired"
    for typ in ["Shock", "Reward"]:
        on = d["cs_on"][(d["cs_type"] == typ) & d["cs_imaged"]]
        d["n" + typ] = len(on)
        d["z" + typ], d["grid"] = M.ca_cue_z(d["F"], d["t"], on)
tab = pd.DataFrame([{"file": d["file"], "genotype": d["gen"], "group": d["grp"], "ROIs": d["F"].shape[1], "frames": d["F"].shape[0],
                     "frame interval s": round(d["dt"], 4), "CS events": len(d["cs_on"]), "imaged CS-shock": d["nShock"],
                     "imaged CS-reward": d["nReward"], "F range": f'{d["F"].min():.1f} to {d["F"].max():.1f}'} for d in DISC])
print(tab.to_string())
print("CS event types per file:", collections.Counter(tuple(sorted(collections.Counter(d["cs_type"].tolist()).items())) for d in DISC))
# Imaged clips per session and how far each CS event is covered by a clip (from before onset to +1, +3, +10 s)
def covered(d, typ, post):
    on = d["cs_on"][d["cs_type"] == typ]
    return int(sum(((d["clip_start"] <= o) & (d["clip_stop"] >= o + post)).any() for o in on))
print(pd.DataFrame([{"file": d["file"], "clips": len(d["clip_start"]), **{f"{typ} to +{p} s": covered(d, typ, p) for typ in ["Shock", "Reward"] for p in [1, 3, 10]}}
                    for d in DISC]).to_string())
cnt = tab.groupby(["genotype", "group"]).agg(mice=("file", "size"), ROIs=("ROIs", "sum"))
cnt["paper neurons"] = [320, 237, 296, 257]; cnt["paper mice"] = [10, 8, 9, 6]
print(cnt)
print("context sessions:", len(CTX), pd.DataFrame([{"file": d["file"], "genotype": d["gen"], "context": d["session_description"], "ROIs": d["F"].shape[1],
      "frame interval s": round(d["dt"], 4), "minutes": round((d["t"][-1] - d["t"][0]) / 60, 1)} for d in CTX]).to_string())
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, "elapsed s", round(time.time() - T0))

Checks of the description against the data.

- Groups. The Neuropixels files say paired or unpaired in the subject description; the Open Ephys
  files say neither. The paper's 4 unpaired ASt mice match the 4 Neuropixels files marked
  unpaired, so the 12 Open Ephys mice are taken as paired (consistent with the data, not shown).
- Regions. Open Ephys electrode groups say ASt (10 mice) or CeA (2). Every Neuropixels electrode
  is labeled ASt or CeA by file, and the units have no region column. They do have `probe_depth`
  (micrometers from the tip, 0 to 3620 in NPX-10), a `label` column (good or mua) and the files
  hold histology images of the probe track, which could support a region per unit; but no file
  maps depth to region, so the TS units the paper analyses cannot be found as the files stand.
  Units labeled good and below 50 Hz are still more than the paper's counts (fig2 cell).
- Units. The Open Ephys units table is the curated set (its description gives quality and spike
  count rules); all its units are kept by the code's 50 Hz rule except where the table shows.
- Cue timing. The CS-shock tones last 20 s, and the CS-reward tones end about 0.4 s after the
  first port entry (median 0.40 s), as the Methods say collection ends the tone. The reward is
  delivered 1 s after onset. In 0 to 28 trials per session the mouse is already in the port at
  onset, so the first entry after onset can come long after the tone ended (5th percentile -8.6
  s); the tone duration is therefore taken as the port latency, with the first entry as the other
  reading. Spikes respond at the true CS onsets and not 45 s earlier, so spikes and trial events
  share a clock.
- Pose clock. In the Open Ephys files the pose is not on the trial clock: in no session is the
  best port-entry lag within 5 s of zero. In four of the six 2018 sessions (3012, 3016, 3024,
  3029) the nose clusters at port entry (spread 7 to 15 px) at a lag of -6.6 s in 3012, -9.8 s in
  3016 and 3024 and +6.6 s in 3029, and in three of them the shock check agrees within 0.6 s; in
  3014 and 3026 the best spread is 60 and 39 px; in the six 2019 sessions (pose at 15 frames/s) no
  lag brings the spread below 81 px. Fitted in each quarter of the session, the lag is constant
  (within 0.8 s, spread 5 to 15 px) in the same four 2018 sessions, so these have a fixed offset
  and no drift; in 3014, 3026 and every 2019 session it changes by 22 to 529 s between quarters,
  so no offset, or offset plus linear drift, maps them. The shock check agrees with the fixed
  offset in 3012, 3016 and 3024 only (2 ASt mice). In 32 of the 34 imaging files the same check
  finds lags of 0.2 to 4 s with a spread of 2.5 to 14 px, as expected from a port LED preceding
  the nose; the exceptions are C6-J588, whose nose stays near the port most of the session, and
  the J933 copy dropped below. The Open Ephys behavior is not used with the spikes or the cues.
- Imaging. Frame interval 1/15 s in every file; the Methods say 15 frames/s downsampled by two.
  ROI counts are 2.4 to 4.0 times the paper's neuron counts: the files hold every CNMF-E ROI, and
  the paper's manual curation is not recorded. Each session has 30 imaged clips (31 in the
  DISC6+TONES sessions, 22 in the truncated G4B2-C8-6860), covering 15 CS-shock and 15 CS-reward
  events (17 to 19 CS-reward in the DISC6+TONES sessions). The Methods say every shock trial and
  every other reward trial were imaged, which would be about 22 of the 45 CS-reward trials; the
  files hold 15. Every event a clip covers to 1 s after onset is covered to 10 s, except one
  CS-reward event of G4B2-C8-6860, so the 10 s requirement drops no trial elsewhere.

## fig2-recorded-neurons (link)

> The dandiset holds the electrophysiology units the paper analyses, with the counts per region
> and group it reports.

Counts of units below 50 Hz (Open Ephys: spike count over recording span; Neuropixels: the
`firing_rate` column), by the region and group the files give.

In [ ]:
rows = []
for d in OE:
    rows.append({"mouse": d["subject_id"], "system": "OE", "group": "paired (assumed)", "region": d["region"], "units<50Hz": len(d["units"]), "has events": True})
npx = INV[INV.kind == "neuropixels"]
for _, r in npx.iterrows():
    rows.append({"mouse": r["file"].split("_ses")[0], "system": "NPX", "group": "unpaired" if "unpaired" in r["subject_description"] else "paired",
                 "region": r["electrode_locations"], "units<50Hz": r["units_below_50Hz"], "has events": bool(r["trials"])})
U = pd.DataFrame(rows); print(U.to_string())
def cnt(q):
    s = U.query(q); return int(s["units<50Hz"].sum()), len(s)
res = {"paired ASt": cnt("group != 'unpaired' and region in ['ASt']"), "paired TS": (0, 0),
       "paired CeA": cnt("region == 'CeA'"), "unpaired ASt": cnt("group == 'unpaired'")}
exp = {"paired ASt": (193, 15), "paired TS": (212, 7), "paired CeA": (145, 8), "unpaired ASt": (71, 4)}
for k in res:
    print(f"{k}: dataset {res[k][0]} units in {res[k][1]} mice; paper {exp[k][0]} neurons in {exp[k][1]} mice")
ev = {k: cnt(q + " and `has events` == True") for k, q in [("paired ASt", "group != 'unpaired' and region in ['ASt']"), ("paired CeA", "region == 'CeA'"), ("unpaired ASt", "group == 'unpaired'")]}
print("usable with trial events:", ev)
gd = INV[INV.kind == "neuropixels"].assign(grp=lambda x: np.where(x.subject_description.str.contains("unpaired"), "unpaired", "paired"))
print("variant, Neuropixels units labeled 'good' below 50 Hz (the files' label column):",
      gd.groupby(["grp", "electrode_locations"]).good_units_below_50Hz.agg(["sum", "size"]).rename(columns={"sum": "units", "size": "mice"}).to_dict("index"))

**Result: not reproduced.** No neuron count matches. The files give 1515 paired "ASt" units in 15
mice against the paper's 193, because each Neuropixels probe's units carry no region; no unit is
labeled TS (paper 212 in 7 mice); 885 CeA units in 6 mice against 145 in 8; 633 unpaired units in
4 mice against 71. The paired ASt and unpaired mouse counts (15 and 4) match. Keeping only the
Neuropixels units labeled good gives 574 paired ASt units (686 with the 112 Open Ephys units)
against 193, 155 CeA (177 with Open Ephys) against 145, and 410 unpaired against 71, so the
label column does not recover the paper's selection either. Only the Open Ephys sessions have
trial events: 112 paired ASt units in 10 mice and 22 CeA units in 2 mice. The claims below that
need paired ASt units are tested on those 10 mice; those that need TS, CeA or unpaired units are
blocked.

## figs9-sustained-only-cs-shock

> Sustained ASt responses occur to CS-shock but not to CS-reward, including CS-reward trials in
> which the reward was not collected for at least 2 s; phasic responses do not differ.

Paired ASt units of the 10 Open Ephys mice (5 of the paper's 15 paired ASt mice are Neuropixels
mice without events). Port latency has two readings: the tone duration, as the Methods say
"Collection of the reward prior to the end of the tone resulted in termination of the tone"
(adopted: the claim's statement is about trials "in which the reward was not collected for at
least 2 s"), and the first port entry at or after CS-reward onset, which misses collections by a
mouse already in the port at onset (data check). The slow group is each neuron's z from the CS-reward trials with
latency over 2 s, in mice that have such trials. Readings: bin 100 ms or 10 ms; the 1 s and 2 s
timepoints as (a) the bin at 1 s and 2 s, (b) the mean from 0.1 s to 1 s and to 2 s.

In [ ]:
def figs9(width, lat_key):
    G = {"CS-shock": [], "CS-reward": [], "CS-reward latency>2s": []}; mice = 0; neurons = 0; per_mouse = []
    for d in AST:
        slow = d["reward_on"][d[lat_key] > 2]
        if len(slow): mice += 1; neurons += len(d["units"])
        for s in d["units"]:
            for k, on in [("CS-shock", d["shock_on"]), ("CS-reward", d["reward_on"]), ("CS-reward latency>2s", slow)]:
                if len(on) == 0: continue
                z, c = M.cue_z(s, on, width)
                b1 = z[np.argmin(abs(c - 1.0 + width / 2))]; b2 = z[np.argmin(abs(c - 2.0 + width / 2))]
                G[k].append([M.window_mean(z, c, M.PHASIC), b1, b2, M.window_mean(z, c, (0.1, 1)), M.window_mean(z, c, (0.1, 2)), d["subject_id"][:12]])
        per_mouse.append((d["subject_id"][:12], len(d["units"]), int((d[lat_key] > 2).sum())))
    return G, mice, neurons, per_mouse
FIGS9 = {}
for (lat_key, lname), width in [(k, w) for k in [("latency_tone", "tone duration"), ("latency", "first port entry")] for w in [0.1, 0.01]]:
    G, mice, neurons, per_mouse = figs9(width, lat_key)
    print(f"latency as {lname}, bin {width*1000:.0f} ms: mice with latency>2 s trials {mice}, neurons {neurons} (paper 9 mice, 96 neurons)")
    if width == 0.1: print("  per mouse (units, slow trials):", per_mouse)
    for j, name in enumerate(["phasic 0-0.1 s", "(a) bin at 1 s", "(a) bin at 2 s", "(b) mean 0.1-1 s", "(b) mean 0.1-2 s"]):
        g = [np.array([r[j] for r in G[k]], float) for k in G]
        F, p = ss.f_oneway(*g); dfw = sum(len(x) for x in g) - 3
        line = f"  {name}: means " + ", ".join(f"{k} {x.mean():.2f}" for k, x in zip(G, g)) + f"; F(2,{dfw}) = {F:.2f}, p = {p:.2g}"
        if j > 0:
            tk = pairwise_tukeyhsd(np.concatenate(g), np.repeat(list(G), [len(x) for x in g]))
            line += "; Tukey " + "; ".join(f"{a} vs {b} diff {m:.2f} p {pp:.3g}" for (a, b), m, pp in zip(
                [(tk.groupsunique[i], tk.groupsunique[k]) for i, k in zip(*tk._multicomp.pairindices)], tk.meandiffs, tk.pvalues))
        print(line)
        FIGS9[(lname, width, name)] = (F, p)
    # by mouse, sustained mean 0.1-2 s
    if width == 0.1 and lat_key == "latency_tone":
        bym = pd.DataFrame([(r[5], k, r[4]) for k in G for r in G[k]], columns=["mouse", "group", "z"]).groupby(["mouse", "group"]).z.mean().unstack().round(2)
        print("  mean z 0.1-2 s by mouse:\n", bym.to_string())

**Result: partially reproduced.** With the port latency taken as the tone duration (adopted),
under both bin sizes the phasic ANOVA is not significant (p 0.38 and 0.34). Under reading (b) of
the timepoints every sustained ANOVA gives p < 0.001 (9.0e-8 to 3.5e-8 at 100 ms, 2.3e-7 and
1.1e-7 at 10 ms), and Tukey's test puts CS-shock above both CS-reward groups (p < 2e-5), while the
two CS-reward groups do not differ (p > 0.89). Under reading (a) the 10 ms bin at 2 s gives p =
0.0013, so reading (b) is the one that meets the criterion at both bin sizes; at 100 ms both
readings do. The first-entry latency gives the same (phasic p 0.24 and 0.23, sustained p < 0.001
in all eight). The slow-trial group does not match the paper's: every one of the 10 mice has
CS-reward trials over 2 s (26 to 86 per mouse; expected 9 mice, within tolerance), giving 112
neurons against 96 from only 10 of the 15 mice, so the paper's selection of these trials must
differ.

## figs10-no-reward-relationship

> ASt responses to CS-reward are not related to reward behavior, across mice or across trials
> grouped by port latency.

A claim of absence, judged on the paper's two tests at p < 0.05. The across-mice test is a one-way
ANOVA over unpaired, low-reward and high-reward mice; no unpaired mouse has trial events, so it
cannot be run (a paired-only comparison of low and high mice is shown as a variant). The
across-trials test is run as the criterion gives it: each paired ASt mouse's CS-reward trials split
into quartiles by port latency (Q1 the fastest), per neuron the phasic z (0 to 100 ms) from the
trials of each quartile, and a repeated measures ANOVA over quartiles with neurons as subjects.
Port latency has two readings: the tone duration (the Methods say collecting the reward ends the
tone, so the tone stops at collection; adopted) and the first port entry at or after onset. Both
bin sizes are run.

In [ ]:
from statsmodels.stats.anova import AnovaRM
def latency_quartiles(lat_key, width):
    rows = []
    for d in AST:
        lat = d[lat_key]; ok = np.isfinite(lat); on = d["reward_on"][ok]
        q = pd.qcut(pd.Series(lat[ok]).rank(method="first"), 4, labels=False).values + 1
        for u, s in enumerate(d["units"]):
            for k in range(1, 5):
                z, c = M.cue_z(s, on[q == k], width)
                rows.append((d["subject_id"][:12], f"{d['subject_id'][:4]}-{u}", k, M.window_mean(z, c, M.PHASIC)))
    df = pd.DataFrame(rows, columns=["mouse", "neuron", "quartile", "z"])
    bad = df.groupby("neuron").z.apply(lambda v: v.isna().any())
    return df[~df.neuron.isin(bad[bad].index)], int(bad.sum())
FIGS10 = {}
for lat_key, lname in [("latency_tone", "tone duration"), ("latency", "first port entry")]:
    for width in [0.1, 0.01]:
        df, dropped = latency_quartiles(lat_key, width)
        a = AnovaRM(df, "z", "neuron", within=["quartile"]).fit().anova_table.iloc[0]
        FIGS10[(lname, width)] = a["Pr > F"]
        print(f"latency as {lname}, bin {width*1000:.0f} ms: neurons {df.neuron.nunique()} (dropped, no baseline variance in a quartile: {dropped}); "
              f"mean phasic z Q1-Q4 {df.groupby('quartile').z.mean().round(2).tolist()}; F({a['Num DF']:.0f},{a['Den DF']:.0f}) = {a['F Value']:.2f}, p = {a['Pr > F']:.3g}")
        if lat_key == "latency_tone" and width == 0.1:
            print("  mean phasic z by mouse and quartile:\n", df.groupby(["mouse", "quartile"]).z.mean().unstack().round(2).to_string())
# Variant for the across-mice test, paired mice only: two clusters by mean latency (tone duration), mouse mean phasic CS-reward z.
mm = pd.DataFrame([{"mouse": d["subject_id"][:12], "mean latency s": d["latency_tone"].mean(),
                    "phasic z": np.nanmean([M.window_mean(*M.cue_z(s, d["reward_on"], 0.1), M.PHASIC) for s in d["units"]])} for d in AST])
for meth in ["average", "ward"]:
    lab = fcluster(linkage(mm[["mean latency s"]].values, meth), 2, "maxclust")
    lo = lab == lab[np.argmin(mm["mean latency s"].values)]
    t = ss.ttest_ind(mm["phasic z"][lo], mm["phasic z"][~lo])
    print(f"variant, {meth} linkage: low-latency mice {int(lo.sum())}, high-latency {int((~lo).sum())}; phasic z {mm['phasic z'][lo].mean():.2f} against {mm['phasic z'][~lo].mean():.2f}, t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
print(mm.round(2).to_string())

**Result: partially reproduced.** The across-trials test can be run and mostly finds no
relationship: with the tone-duration latency and 100 ms bins, the repeated measures ANOVA of
phasic CS-reward z over latency quartiles gives F(3,330) = 1.81, p = 0.14 (111 neurons, 1
dropped), and with the first-entry latency p = 0.185 and 0.18 at 100 and 10 ms. With the
tone-duration latency and 10 ms bins it gives p = 0.047, just below 0.05, with the slowest
quartile highest (1.47 against 1.21 to 1.31). The across-mice test needs the unpaired mice and
cannot be run; among the paired mice, the low-latency and high-latency clusters (7 and 3 mice,
both linkages) do not differ (p = 0.60). The claim holds in part: one of its two tests cannot be
run, and the other is not significant under three of four readings.

## fig2o-region-differs-cs-reward

> Responses to CS-reward also differ between regions, and the ASt has robust phasic responses to
> CS-reward as well as to CS-shock.

The region ANOVAs need TS units, which no file labels, and the CeA units with trial events are 22
from 2 of the paper's 8 CeA mice; they are not run as the paper's test. The ASt part is run: the
one-sample t-test of the ASt phasic CS-reward z against 0 (the criterion's choice), under both bin
sizes, on the 112 paired ASt units. For the record, the ASt against CeA comparison is printed. The
same per-neuron table (z per CS and window, and the responsive class under both readings of the
signed-rank inputs) serves the blocked claims fig2o-ast-greater-cs-shock-responses and
fig2p-ast-more-responsive at the end.

In [ ]:
def trial_rates(s, on, win):
    return np.array([((s >= o + win[0]) & (s < o + win[1])).sum() / (win[1] - win[0]) for o in on])
def response_class(s, on, win, z, reading):
    """Opening comment of the claims file: signed-rank test at 0.01 and mean z above +2 or below -2.
    Reading (a) against the same trial's 20 s baseline, (b) against an equal-length window before onset."""
    r = trial_rates(s, on, win)
    b = trial_rates(s, on, (-20, 0) if reading == "a" else (-(win[1] - win[0]), 0))
    p = 1.0 if np.all(r == b) else ss.wilcoxon(r, b).pvalue
    return 1 if (p < 0.01 and z > 2) else (-1 if (p < 0.01 and z < -2) else 0)
rows = []
for width in [0.1, 0.01]:
    for d in OE:
        for u, s in enumerate(d["units"]):
            r = {"width": width, "region": d["region"], "mouse": d["subject_id"][:12], "unit": u}
            for cs, on in [("shock", d["shock_on"]), ("reward", d["reward_on"])]:
                z, c = M.cue_z(s, on, width)
                for wn, win in [("phasic", M.PHASIC), ("sustained", M.SUSTAINED)]:
                    r[f"{cs} {wn}"] = zz = M.window_mean(z, c, win)
                    for rd in "ab":
                        r[f"{cs} {wn} class {rd}"] = response_class(s, on, win, zz, rd)
            rows.append(r)
R2O = pd.DataFrame(rows)
for width in [0.1, 0.01]:
    g = R2O[R2O.width == width]
    a = g[g.region == "ASt"]["reward phasic"].dropna(); t = ss.ttest_1samp(a, 0)
    print(f"bin {width*1000:.0f} ms: ASt phasic CS-reward z mean {a.mean():.2f} (n {len(a)}, range {a.min():.1f} to {a.max():.1f}), one-sample t({len(a)-1}) = {t.statistic:.2f}, p = {t.pvalue:.2g}")
    print("  by mouse:", g[g.region == "ASt"].groupby("mouse")["reward phasic"].mean().round(2).to_dict())
    for col in ["reward phasic", "reward sustained"]:
        x, y = g[g.region == "ASt"][col].dropna(), g[g.region == "CeA"][col].dropna(); tt = ss.ttest_ind(x, y)
        print(f"  for the record, ASt against CeA, CS-{col}: {x.mean():.2f} (n {len(x)}) against {y.mean():.2f} (n {len(y)}), F(1,{len(x)+len(y)-2}) = {tt.statistic**2:.2f}, p = {tt.pvalue:.2g}")

**Result: partially reproduced.** The ASt part holds: the ASt phasic CS-reward z is above 0
(mean 7.09, t(111) = 7.79, p = 3.9e-12 at 100 ms; 2.56, p = 7.4e-12 at 10 ms), in all 10 mice
(mouse means 0.12 to 13.04). The region part cannot be run as the paper's test: there are no TS
units and 22 CeA units from 2 of the 8 CeA mice. For the record, ASt against CeA gives p = 0.061
and 0.039 for the phasic and 0.20 and 0.27 for the sustained CS-reward z (100 and 10 ms), none
near the criterion's p < 0.0001.

## fig2q-trajectories-diverge

> ASt population trajectories for CS-shock and CS-reward diverge from cue onset.

Code's procedure (trajectories_rev_2025.py): 100 ms bins, -20 to 40 s; each neuron z-scored over
all its trials and bins; averaged per CS; neurons of all paired Open Ephys ASt mice stacked. The
Methods' reading fits one PCA on paired and unpaired animals; no unpaired animal has events, so
the PCA is fitted on the paired animals. In the complete PC space the PCA is a rotation of the
centred neuron space, so the distance between the paired trajectories is the same whatever rows
the PCA is fitted on: the complete-space value is the Methods' reading as well as the code's.
The distance in the components that explain 90% of variance (code) depends on the fit.

In [ ]:
W = 0.1; Ms, Mr = [], []
for d in AST:
    for s in d["units"]:
        cs, c = M.binned(s, d["shock_on"], -20, 40, W); cr, _ = M.binned(s, d["reward_on"], -20, 40, W)
        a = np.concatenate([cs, cr]); mu, sd = a.mean(), a.std()
        if sd == 0: continue
        Ms.append(((cs - mu) / sd).mean(0)); Mr.append(((cr - mu) / sd).mean(0))
Ms, Mr = np.array(Ms), np.array(Mr)
pca = PCA().fit(np.concatenate([Ms.T, Mr.T])); Ps, Pr = pca.transform(Ms.T), pca.transform(Mr.T)
k90 = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9) + 1)
for name, dist in [("complete PC space", np.linalg.norm(Ps - Pr, axis=1)), (f"first {k90} PCs (90% variance)", np.linalg.norm(Ps[:, :k90] - Pr[:, :k90], axis=1))]:
    print(f"{name}: neurons {Ms.shape[0]}; mean distance 0-18 s {dist[(c > 0) & (c < 18)].mean():.2f}; "
          f"baseline -20-0 s max {dist[c < 0].max():.2f}, mean {dist[c < 0].mean():.2f}; distance range {dist.min():.2f} to {dist.max():.2f}")
# by mouse, complete space (each mouse's own neurons)
i0 = 0
for d in AST:
    n = len(d["units"]); A_, B_ = Ms[i0:i0 + n], Mr[i0:i0 + n]; i0 += n
    dd = np.linalg.norm(A_ - B_, axis=0)
    print(f"  {d['subject_id'][:12]}: {n} neurons, cue mean {dd[(c > 0) & (c < 18)].mean():.2f}, baseline max {dd[c < 0].max():.2f}")
fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(c, np.linalg.norm(Ps - Pr, axis=1)); ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("time from CS onset (s)"); ax.set_ylabel("shock-reward distance"); plt.show()

**Result: reproduced.** In the complete PC space the mean CS-shock to CS-reward distance over 0 to
18 s is 7.36, above the largest baseline distance, 2.87; in the 41 components that explain 90%
of the variance it is 7.16 against 1.94. Within single mice the cue mean exceeds the baseline
maximum in 3 of the 10 (3016, 3024 and 7008); the claim, like the paper's
figure, is on the stacked population. The complete-space distance does not depend on whether
the PCA is fitted with or without unpaired animals, so it is the Methods' reading; the 90%
variant is fitted on paired mice only.

## figs8-single-trial-divergence

> In single trial pairs from paired mice, CS-shock and CS-reward population trajectories are less
> similar during the cue than pairs of the same CS; unpaired mice do not show this.

Methods: per animal, 100 random draws of trial pairs; each trial's population vector (100 ms bins,
standard choice from the code) minus its mean over -20 to 0 s; cosine similarity per bin, averaged
over 0 to 18 s. Animals with at least 10 units and 30 trials per CS (code). Readings of the
Wilcoxon test: (a) per-animal means, shock-reward against the mean of the same-CS pairs; (b)
against each same-CS comparison; (c) iterations pooled. Unpaired animals have no events.

In [ ]:
def cos_animal(d, rng):
    S = np.stack([M.binned(s, d["shock_on"], -20, 40, 0.1)[0] for s in d["units"]], 2) / 0.1
    R = np.stack([M.binned(s, d["reward_on"], -20, 40, 0.1)[0] for s in d["units"]], 2) / 0.1
    c = M.binned(d["units"][0], d["shock_on"], -20, 40, 0.1)[1]
    S = S - S[:, c < 0].mean(1, keepdims=True); R = R - R[:, c < 0].mean(1, keepdims=True)
    w = (c > 0) & (c < 18)
    def cs(a, b):
        with np.errstate(invalid="ignore", divide="ignore"):
            v = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
        return np.nanmean(v[w])
    out = []
    for _ in range(100):
        i, j = rng.choice(len(S), 2, replace=False); k, l = rng.choice(len(R), 2, replace=False)
        out.append((cs(S[i], S[j]), cs(R[k], R[l]), cs(S[i], R[k])))
    return np.array(out)
SEL8 = [d for d in AST if len(d["units"]) >= 10 and len(d["shock_on"]) >= 30 and len(d["reward_on"]) >= 30]
print("animals:", [(d["subject_id"][:12], len(d["units"])) for d in SEL8], "(excluded for <30 CS-shock trials:",
      [d["subject_id"][:12] for d in AST if len(d["units"]) >= 10 and len(d["shock_on"]) < 30], ")")
print("smallest two-sided Wilcoxon signed-rank p with", len(SEL8), "animals:", ss.wilcoxon(np.arange(1, len(SEL8) + 1), np.zeros(len(SEL8))).pvalue)
for off in [0, 1000, 2000, 3000, 4000]:
    rng = np.random.default_rng(seed("figs8-single-trial-divergence", off))
    res = [cos_animal(d, rng) for d in SEL8]
    m = np.array([r.mean(0) for r in res]); P = np.concatenate(res)
    a = ss.wilcoxon(m[:, 2], m[:, :2].mean(1)); b1 = ss.wilcoxon(m[:, 2], m[:, 0]); b2 = ss.wilcoxon(m[:, 2], m[:, 1])
    cpool = ss.wilcoxon(P[:, 2], P[:, :2].mean(1))
    print(f"seed +{off}: pooled means SS {P[:,0].mean():.4f} RR {P[:,1].mean():.4f} SR {P[:,2].mean():.4f}; "
          f"(a) p {a.pvalue:.3g}; (b) vs SS p {b1.pvalue:.3g}, vs RR p {b2.pvalue:.3g}; (c) pooled n={len(P)} W={cpool.statistic:.0f} p {cpool.pvalue:.3g}")
    if off == 0:
        print(pd.DataFrame(m, columns=["SS", "RR", "SR"], index=[d["subject_id"][:12] for d in SEL8]).round(3).to_string())

**Result: partially reproduced.** In the five paired animals that meet the code's selection,
shock-reward similarity (pooled mean about -0.01) is below shock-shock (0.07) and reward-reward
(0.07); with the iterations pooled, reading (c), p is below 1e-47 under all five seeds. Reading
(c) treats the 100 resamples of the same trials as independent, so its p overstates the
evidence. With five animals the per-animal readings (a) and (b) cannot reach p < 0.01: the
smallest two-sided Wilcoxon p for five pairs is 0.0625, and they give p 0.06 to 0.13. The
unpaired half of the claim cannot be tested: no unpaired animal has trial events.

## fig3h-cs-decoded-paired-only

> CS identity can be decoded from ASt activity above chance during the cue in paired but not
> unpaired mice.

The code's decoder (fig4h_decoding_workflow.py), as the criterion gives it, on the paired Open
Ephys mice with more than 10 units below 50 Hz. Mann-Whitney U test of the mice's accuracies
against their chance values (legend), with the code's Wilcoxon signed-rank test as a variant.

In [ ]:
def decode(d, rng, nshuf=20):
    n = min(29, len(d["shock_on"]), len(d["reward_on"]))
    so = np.sort(rng.choice(d["shock_on"], n, replace=False)); ro = np.sort(rng.choice(d["reward_on"], n, replace=False))
    X = np.concatenate([np.stack([M.binned(s, on, -20, 40, 0.1)[0] for s in d["units"]], 2) for on in (so, ro)])
    c = M.binned(d["units"][0], so, -20, 40, 0.1)[1]
    y = np.r_[np.ones(n), np.zeros(n)]; nb = X.shape[1]
    acc = np.zeros(nb); ch = np.zeros(nb)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=int(rng.integers(2**31))).split(np.zeros(len(y)), y):
        Xtr = X[tr].reshape(-1, X.shape[2]); sc = StandardScaler().fit(Xtr); Xtr = sc.transform(Xtr)
        Xte = sc.transform(X[te].reshape(-1, X.shape[2]))
        m = LogisticRegression(solver="liblinear").fit(Xtr, np.repeat(y[tr], nb))
        acc += (m.predict(Xte).reshape(len(te), nb) == y[te][:, None]).sum(0)
        for _ in range(nshuf):
            m2 = LogisticRegression(solver="liblinear").fit(Xtr, np.repeat(rng.permutation(y[tr]), nb))
            ch += (m2.predict(Xte).reshape(len(te), nb) == y[te][:, None]).sum(0) / nshuf
    w = (c > 0) & (c < 18)
    return (acc / len(y))[w].mean(), (ch / len(y))[w].mean()
SEL3H = [d for d in AST if len(d["units"]) > 10]
print("mice:", [(d["subject_id"][:12], len(d["units"])) for d in SEL3H])
for off in [0, 1000, 2000, 3000, 4000]:
    rng = np.random.default_rng(seed("fig3h-cs-decoded-paired-only", off))
    r = np.array([decode(d, rng) for d in SEL3H])
    mw2 = ss.mannwhitneyu(r[:, 0], r[:, 1]); mw1 = ss.mannwhitneyu(r[:, 0], r[:, 1], alternative="greater"); wx = ss.wilcoxon(r[:, 0], r[:, 1])
    print(f"seed +{off}: accuracy {np.round(r[:,0],3).tolist()} chance {np.round(r[:,1],3).tolist()}; "
          f"MWU U={mw2.statistic:.0f} two-sided p {mw2.pvalue:.4f}, one-sided p {mw1.pvalue:.4f}; Wilcoxon W={wx.statistic:.0f} p {wx.pvalue:.4f}")

**Result: not reproduced.** Accuracy over 0 to 18 s is above shuffled chance (about 0.5) in four
of the six paired mice (0.66 to 0.89) and near it in two, 3026 (0.47 to 0.49 over the five seeds)
and 6197 (0.53 to 0.60), against shuffled chance of 0.49 to 0.50 in both; no per-mouse test was run. The Mann-Whitney U test
of accuracies against chance gives p = 0.026 two-sided under the claim's seed and 0.015 to 0.065
under the four others, never below the criterion's 0.01 (one-sided 0.008 to 0.033). The code's
Wilcoxon variant gives p = 0.0625. The unpaired half cannot be tested: no unpaired mouse has
trial events.

## figs13-imaging-mice-discriminate

> Paired Drd1a-Cre and Drd2-Cre mice in the imaging experiments show more defensive behavior to
> CS-shock than to CS-reward, and unpaired mice do not.

Per frame, freezing and darting from the poses (helper `behaviour_frames`; freezing rule "max",
every body point moved less than 0.08 cm, or "sum", the summed displacement below 0.08 cm;
darting when the haunch reaches 45 cm/s). The pixel scale is taken from the extent of the body
points against the 30 x 40 cm chamber (assumption). Defensive behavior per trial is the share of
the 10 s tone spent freezing plus the share spent darting; all 60 CS events of each session.

In [ ]:
BEH13 = []
for d in DISC:
    ppc = M.pixels_per_cm(d["pose"]); fps = 1 / np.median(np.diff(d["pose_t"]))
    r = {"file": d["file"], "genotype": d["gen"], "group": d["grp"], "px/cm": round(ppc, 1)}
    for rule in ["max", "sum"]:
        frz, dart = M.behaviour_frames(d["pose"], ppc, fps=fps, freeze_rule=rule)
        for typ in ["Shock", "Reward"]:
            on = d["cs_on"][d["cs_type"] == typ]
            r[f"{rule} {typ}"] = np.nanmean(M.window_fraction(d["pose_t"], frz, on, 10) + M.window_fraction(d["pose_t"], dart, on, 10))
    BEH13.append(r)
BEH13 = pd.DataFrame(BEH13); print(BEH13.round(3).to_string())
for rule in ["max", "sum"]:
    for (gen, grp), g in BEH13.groupby(["genotype", "group"]):
        t = ss.ttest_rel(g[f"{rule} Shock"], g[f"{rule} Reward"])
        print(f"{rule}: {gen} {grp}, n={len(g)}: CS-shock {g[f'{rule} Shock'].mean():.3f}, CS-reward {g[f'{rule} Reward'].mean():.3f}, t({len(g)-1}) = {t.statistic:.2f}, p = {t.pvalue:.4f}")
# Structure of the frame-to-frame displacement: the largest displacement over body points, in pixels.
# The freezing threshold (0.08 cm) is about 1.6 px at the inferred scale.
mx = np.concatenate([M.max_displacement_px(d["pose"]) for d in DISC]); mx = mx[np.isfinite(mx)]
print("largest point displacement per frame, all discrimination sessions, share of frames:",
      {k: round(float(v), 4) for k, v in zip(["<0.25 px", "0.25-0.5", "0.5-1", "1-1.6", "1.6-2", ">2"], np.histogram(mx, [0, 0.25, 0.5, 1, 1.6, 2, np.inf])[0] / len(mx))})
print("exactly repeated poses (all points equal to the previous frame):", round(float(np.mean(mx == 0)), 4))
# Freezing and darting separately, and the median-displacement variant (not the paper's rule).
rows = []
for d in DISC:
    ppc = M.pixels_per_cm(d["pose"]); fps = 1 / np.median(np.diff(d["pose_t"]))
    frz, dart = M.behaviour_frames(d["pose"], ppc, fps=fps)
    frz_med = M.behaviour_frames(d["pose"], ppc, fps=fps, freeze_rule="median")[0]
    r = {"file": d["file"], "genotype": d["gen"], "group": d["grp"]}
    for typ in ["Shock", "Reward"]:
        on = d["cs_on"][d["cs_type"] == typ]
        r[f"freeze {typ}"] = np.nanmean(M.window_fraction(d["pose_t"], frz, on, 10))
        r[f"dart {typ}"] = np.nanmean(M.window_fraction(d["pose_t"], dart, on, 10))
        r[f"median rule {typ}"] = np.nanmean(M.window_fraction(d["pose_t"], frz_med, on, 10) + M.window_fraction(d["pose_t"], dart, on, 10))
    rows.append(r)
B13 = pd.DataFrame(rows)
for m in ["freeze", "dart", "median rule"]:
    for (gen, grp), g in B13.groupby(["genotype", "group"]):
        t = ss.ttest_rel(g[f"{m} Shock"], g[f"{m} Reward"])
        print(f"{m}: {gen} {grp}, n={len(g)}: CS-shock {g[f'{m} Shock'].mean():.3f}, CS-reward {g[f'{m} Reward'].mean():.3f}, t({len(g)-1}) = {t.statistic:.2f}, p = {t.pvalue:.4f}")


**Result: reproduced.** Under both freezing rules, paired mice show more defensive behavior to
CS-shock than to CS-reward (Drd1a-Cre, 10 mice, t(9) = 3.6, p = 0.006; Drd2-Cre, 9 mice,
t(8) = 2.4, p = 0.045), and unpaired mice do not (Drd1a-Cre p = 0.80, Drd2-Cre p 0.51 to 0.52).
The difference comes from darting (paired Drd1a-Cre 0.032 against 0.007, p < 0.0001; Drd2-Cre
0.031 against 0.001, p = 0.0016); freezing alone does not differ between the cues (p = 0.14 and
0.37). The paper's freezing rule counts only frames whose pose repeats the previous one: the
largest point displacement is below 0.5 px in 19.6% of frames, above 1.6 px in the rest, and
never between 0.5 and 1.6 px, so the threshold (about 1.6 px) and the pixel scale do not change
which frames count. Whether those frames are still frames or repeated video frames the files do
not say. The median-displacement variant, which also counts the mouse still at the port, finds
no cue difference (p 0.105 to 0.71). The outcome rests on the paper's rule and on the assumed
pixel scale (18.8 to 20.7 px/cm), which sets the darting threshold.

## fig5j-drd2-cs-shock-responses, fig5h-drd1-no-difference, fig5k-drd2-cs-reward-no-difference

Calcium response per ROI (helper `ca_cue_z`): raw CNMF-E fluorescence averaged over the imaged
trials of a CS type, z-scored by the 10 s before onset; short window 0 to 1 s, long 1 to 10 s;
two-sample Student's t-tests of paired against unpaired ROIs. Every ROI in the files is used: the
paper's manual curation is not recorded in them. Variants: the deconvolved trace; the trace with
a linear trend over the baseline removed (helper `baseline_detrend`, not the paper's method), to
see whether a decay within the clips drives the differences; and, for fig5h, the unpaired group
without G4B2-C8-6860 (1 imaged CS-shock trial) and G2B1-C4-6886 (1 ROI). Mouse-level tests are
shown for each claim.

In [ ]:
def ttests(gen, typ, key="z", skip=()):
    out = {}
    for win, wname in [(M.SHORT, "short"), (M.LONG, "long")]:
        v = {grp: np.concatenate([M.window_mean(d[key + typ], d["grid"], win) for d in DISC if d["gen"] == gen and d["grp"] == grp and not any(s in d["file"] for s in skip)]) for grp in ["paired", "unpaired"]}
        a, b = v["paired"][np.isfinite(v["paired"])], v["unpaired"][np.isfinite(v["unpaired"])]
        t = ss.ttest_ind(a, b)
        out[wname] = (len(a), len(b), a.mean(), b.mean(), t.statistic, t.pvalue)
        print(f"  {gen} CS-{typ.lower()} {wname}: paired n={len(a)} mean {a.mean():.3f} (range {a.min():.1f} to {a.max():.1f}), unpaired n={len(b)} mean {b.mean():.3f}; "
              f"t({len(a)+len(b)-2}) = {t.statistic:.2f}, p = {t.pvalue:.3g}; dropped (no baseline) {np.sum(~np.isfinite(v['paired']))+np.sum(~np.isfinite(v['unpaired']))}")
    return out
def by_mouse(gen, typ):
    rows = [{"file": d["file"], "group": d["grp"], "ROIs": d["F"].shape[1], "short": np.nanmean(M.window_mean(d["z" + typ], d["grid"], M.SHORT)),
             "long": np.nanmean(M.window_mean(d["z" + typ], d["grid"], M.LONG))} for d in DISC if d["gen"] == gen]
    df = pd.DataFrame(rows); print(df.round(3).to_string())
    for w in ["short", "long"]:
        t = ss.ttest_ind(df[df.group == "paired"][w], df[df.group == "unpaired"][w])
        print(f"  mouse-level {w}: t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
for d in DISC:
    for typ in ["Shock", "Reward"]:
        d["dz" + typ] = M.ca_cue_z(d["D"], d["t"], d["cs_on"][(d["cs_type"] == typ) & d["cs_imaged"]])[0]
        d["tz" + typ] = M.baseline_detrend(d["z" + typ], d["grid"])
# Does fluorescence decay within the clips? Mean z over ROIs in windows of the baseline and the cue,
# by genotype, group and CS. A decay shows as baseline z falling from the start of the clip to onset.
rows = []
for d in DISC:
    for typ in ["Shock", "Reward"]:
        for wname, w in [("-10 to -7 s", (-10, -7)), ("-3 to 0 s", (-3, 0)), ("0 to 1 s", (0, 1)), ("1 to 5 s", (1, 5)), ("5 to 10 s", (5, 10))]:
            v = M.window_mean(d["z" + typ], d["grid"], w)
            rows.append({"genotype": d["gen"], "group": d["grp"], "CS": typ, "window": wname, "sum": np.nansum(v), "n": np.isfinite(v).sum()})
dec = pd.DataFrame(rows).groupby(["genotype", "group", "CS", "window"], sort=False)[["sum", "n"]].sum()
print("mean z over ROIs by window:\n", (dec["sum"] / dec["n"]).unstack().round(2).to_string())

In [ ]:
print("Drd2 CS-shock (fig5j), raw fluorescence:"); T5J = ttests("Drd2", "Shock")
print("variant, deconvolved trace:"); ttests("Drd2", "Shock", "dz")
print("variant, baseline trend removed:"); ttests("Drd2", "Shock", "tz")
by_mouse("Drd2", "Shock")


**Result: partially reproduced.** Drd2+ ROIs of paired mice respond more to CS-shock than those
of unpaired mice in both windows (short t(1812) = 6.2, p < 1e-9; long t(1812) = 5.0, p < 1e-6),
but the counts are far outside tolerance: 1189 paired and 625 unpaired ROIs against the paper's
296 and 257, because the files hold every CNMF-E ROI and not the curated neurons. The
deconvolved variant agrees (p < 1e-5), and so does the variant with the baseline trend removed
(p = 4.9e-9 and 0.0044). The fluorescence decays within the clips in every group alike (mean
baseline z falls from 0.27 to 0.36 at -10 to -7 s to -0.19 to -0.28 at -3 to 0 s), so the
decay does not explain the group difference. Across mice (9 and 6) the difference is not
significant (p 0.17 and 0.16).

In [ ]:
print("Drd1a CS-shock and CS-reward (fig5h), raw fluorescence:"); T5H = {t: ttests("Drd1a", t) for t in ["Shock", "Reward"]}
print("variant, deconvolved trace:"); [ttests("Drd1a", t, "dz") for t in ["Shock", "Reward"]]
print("variant, baseline trend removed:"); [ttests("Drd1a", t, "tz") for t in ["Shock", "Reward"]]
SKIP = ("G4B2-C8-6860", "G2B1-C4-6886")
print("variant, without", SKIP, "(1 imaged CS-shock trial in a truncated session; 1 ROI):"); [ttests("Drd1a", t, skip=SKIP) for t in ["Shock", "Reward"]]
by_mouse("Drd1a", "Shock"); by_mouse("Drd1a", "Reward")


**Result: not reproduced.** All four of the paper's tests find a difference on the deposited
ROIs: Drd1a+ paired ROIs (847) are above unpaired ones (608) for CS-shock short (p = 0.0016) and
long (p < 1e-4) and for CS-reward short (p = 0.0018) and long (p = 0.0032), with mean z
differences of 0.2 to 0.4. Without the two thin unpaired sessions all four
remain (p 0.0015 to 3.8e-6), and with the baseline trend removed three of four remain (p 0.0092
to 0.038; CS-reward long p = 0.16). Across mice (10 and 8) nothing differs (CS-shock p 0.89 and
0.90, CS-reward p 0.73 and 0.50). The paper had 320 and 237 curated neurons; these are all
CNMF-E ROIs, about 2.6 times as many, and the neuron-level test is judged as the paper ran it.

In [ ]:
print("Drd2 CS-reward (fig5k), raw fluorescence:"); T5K = ttests("Drd2", "Reward")
print("variant, deconvolved trace:"); ttests("Drd2", "Reward", "dz")
print("variant, baseline trend removed:"); ttests("Drd2", "Reward", "tz")
by_mouse("Drd2", "Reward")


**Result: not reproduced.** The short-window test finds a difference (t(1812) = 2.1, p = 0.034,
paired mean -0.27 against unpaired -0.40); the long window does not (p = 0.61). The effect is
small (difference 0.13 in mean z) and depends on the trace: with the deconvolved trace neither
window differs (p 0.31 and 0.15), with the baseline trend removed the short window does (p =
0.0023). Across mice (9 and 6) neither window differs (p 0.36 and 0.33). This is on all 1814
CNMF-E ROIs, against the paper's 553 curated neurons.

## fig5m-drd2-rich-clusters-cs-shock

> Clustering imaged neurons of paired Drd1a-Cre and Drd2-Cre mice, the clusters richest in Drd2+
> neurons respond to CS-shock or are inhibited by CS-reward ...

Readings named by the criterion: linkage average or Ward; span the 10 s cue or the imaged window
(-9.5 to 10 s, as the clips start about 9.6 s before onset). "Soft normalized to a maximum z of
10" has two readings: z / (max|z| + 10), and clipping at plus or minus 10. Nine clusters; Drd2
share normalized for the number of each type; Spearman correlation of that share with the
cluster's mean CS-shock z over 0 to 10 s. In the window reading, bins before the latest clip start
(-8.67 s) are dropped so that every ROI is kept; a variant that keeps the bins from -9.5 s and
drops the ROIs whose clips start later is shown after it.

In [ ]:
PD = [d for d in DISC if d["grp"] == "paired"]
def on_grid(d, typ, tg):
    return np.array([np.interp(tg, d["grid"], r) for r in d["z" + typ]])
def cluster_rho(X, gen, tg_sh, nsh, norm, meth):
    N2, N1 = (gen == "Drd2").sum(), (gen == "Drd1a").sum()
    Xn = X / (np.abs(X).max(1, keepdims=True) + 10) if norm.startswith("z/") else np.clip(X, -10, 10)
    lab = fcluster(linkage(Xn, meth), 9, "maxclust")
    share, resp, sizes = [], [], []
    for c in np.unique(lab):
        m = lab == c; f2 = (gen[m] == "Drd2").sum() / N2; f1 = (gen[m] == "Drd1a").sum() / N1
        share.append(f2 / (f1 + f2)); sizes.append(int(m.sum())); resp.append(X[m][:, :nsh][:, tg_sh > 0].mean())
    r = ss.spearmanr(share, resp)
    return sizes, r.statistic, r.pvalue
RES5M = []
for span, tg in [("cue 0-10 s", np.arange(0, 10, 1 / 15) + 1 / 30), ("window -9.5-10 s", np.arange(-9.5, 10, 1 / 15) + 1 / 30)]:
    X = np.vstack([np.hstack([on_grid(d, "Shock", tg), on_grid(d, "Reward", tg)]) for d in PD])
    gen = np.concatenate([[d["gen"]] * d["F"].shape[1] for d in PD])
    # bins where some session's clips start later than the span hold no frame; drop those bins, keep every ROI
    ok = np.tile(np.isfinite(np.vstack([on_grid(d, "Shock", tg) for d in PD])).all(0), 2) & np.isfinite(X).all(0)
    keep_sh = ok[:len(tg)]; X = X[:, ok]; tg_sh = tg[keep_sh]; nsh = int(keep_sh.sum())
    print(f"{span}: bins kept {nsh} of {len(tg)} per CS, from {tg_sh.min():.2f} s; ROIs with NaN left {int((~np.isfinite(X)).any(1).sum())}")
    N2, N1 = (gen == "Drd2").sum(), (gen == "Drd1a").sum()
    for norm in ["z/(max|z|+10)", "clip at 10"]:
        for meth in ["average", "ward"]:
            sizes, rho, pv = cluster_rho(X, gen, tg_sh, nsh, norm, meth)
            RES5M.append((span, norm, meth, len(X), sizes, rho, pv))
            print(f"{span}, {norm}, {meth}: ROIs {len(X)} (Drd2 {N2}, Drd1a {N1}), cluster sizes {sizes}, rho = {rho:.2f}, p = {pv:.3f}")

In [ ]:
# Variant (asked for in verification): the window from -9.5 s with every bin kept, dropping the ROIs whose
# clips start later and so hold no frame there. The start is then moved by 0.1 to 0.5 s to show how stable it is.
RES5M_DROP = []
for start in [-9.5, -9.4, -9.2, -9.0]:
    tg = np.arange(start, 10, 1 / 15) + 1 / 30
    late = [d for d in PD if not (np.isfinite(on_grid(d, "Shock", tg)).all() and np.isfinite(on_grid(d, "Reward", tg)).all())]
    PK = [d for d in PD if d not in late]
    X = np.vstack([np.hstack([on_grid(d, "Shock", tg), on_grid(d, "Reward", tg)]) for d in PK])
    gen = np.concatenate([[d["gen"]] * d["F"].shape[1] for d in PK])
    N2, N1 = (gen == "Drd2").sum(), (gen == "Drd1a").sum()
    print(f"from {start} s: dropped {[(d['file'][:30], d['gen'], d['F'].shape[1]) for d in late]}; ROIs {len(X)} (Drd2 {N2}, Drd1a {N1}), NaN left {int((~np.isfinite(X)).sum())}")
    for norm in ["z/(max|z|+10)", "clip at 10"]:
        for meth in ["average", "ward"]:
            sizes, rho, pv = cluster_rho(X, gen, tg, len(tg), norm, meth)
            RES5M_DROP.append((start, norm, meth, len(X), sizes, rho, pv))
            print(f"  {norm}, {meth}: cluster sizes {sizes}, rho = {rho:.2f}, p = {pv:.3f}")


**Result: not reproduced.** Under none of the eight readings is the Spearman correlation between
a cluster's normalized Drd2+ share and its mean CS-shock z positive at p < 0.05 (rho -0.12 to
0.47, p 0.20 to 0.97). Average linkage puts most ROIs in one or two clusters. This is on all CNMF-E ROIs
of paired mice, not the curated neurons.

The variant that keeps the window from -9.5 s and drops the ROIs whose clips start later gives
rho = 0.76, p = 0.017 under one combination (z/(max|z|+10), average linkage) and p 0.29 to 0.58
under the other three. It is not taken as a reading of the criterion, which asks for every neuron
of paired mice: it drops 233 of the 847 Drd1a+ ROIs (C5-6933 and K008, 2 of the 10 Drd1a-Cre
mice), which changes the Drd2+ share that ranks the clusters. The result also depends on the
start: with the same ROIs dropped and the window starting at -9.4 s, the same combination gives
p = 0.22, and from -9.2 or -9.0 s (only K008 dropped) the lowest p is 0.16. The
outcome is kept.


## figs15-no-context-encoding

> After contextual fear conditioning, neither Drd1a+ nor Drd2+ ASt neurons change their activity
> in the conditioned context compared with an unconditioned one, although the mice freeze more in
> the conditioned context.

Per mouse and context: mean raw fluorescence over ROIs and frames; transient peak rate, peaks
above the trace's mean plus 2 SD (scipy find_peaks), per ROI per second; estimated spike rate from
the deconvolved trace, as frames with a nonzero value per second, and as the variant its mean
value. Freezing share of the session from the poses as above. Paired t-tests across mice.

In [ ]:
rows = []
for d in CTX:
    mouse = d["path"].split("-Day4")[0].split("-CFCD4")[0].replace("sub-", "")
    cond = "A" if "Context A" in d["session_description"] else "B"
    dur = d["t"][-1] - d["t"][0]
    pk = [len(find_peaks(r, height=r.mean() + 2 * r.std())[0]) / dur for r in d["F"].T]
    ppc = M.pixels_per_cm(d["pose"]); fps = 1 / np.median(np.diff(d["pose_t"]))
    r = {"mouse": mouse, "context": cond, "genotype": d["gen"], "ROIs": d["F"].shape[1], "minutes": round(dur / 60, 1),
         "raw F": d["F"].mean(), "peaks/s": np.mean(pk), "deconv events/s": np.mean((d["D"] > 0).sum(0) / dur),
         "deconv mean": d["D"].mean(), "px/cm": ppc}
    for rule in ["max", "sum"]:
        r[f"freeze {rule}"] = M.behaviour_frames(d["pose"], ppc, fps=fps, freeze_rule=rule)[0].mean()
    r["freeze max, px/cm 14"] = M.behaviour_frames(d["pose"], 14.0, fps=fps, freeze_rule="max")[0].mean()
    r["freeze median (variant)"] = M.behaviour_frames(d["pose"], ppc, fps=fps, freeze_rule="median")[0].mean()
    r["dart"] = M.behaviour_frames(d["pose"], ppc, fps=fps)[1].mean()
    mx = M.max_displacement_px(d["pose"])
    r["frames <0.5 px"] = np.mean(mx < 0.5); r["frames 0.5-1.6 px"] = np.mean((mx >= 0.5) & (mx < 1.6))
    rows.append(r)
CT = pd.DataFrame(rows).sort_values(["genotype", "mouse", "context"]); print(CT.round(3).to_string())
print("px/cm range in context sessions:", CT["px/cm"].min().round(1), "to", CT["px/cm"].max().round(1), "(the 30 x 40 cm assumption is less secure here)")
for gen in ["Drd1a", "Drd2"]:
    a = CT[(CT.genotype == gen) & (CT.context == "A")].set_index("mouse"); b = CT[(CT.genotype == gen) & (CT.context == "B")].set_index("mouse").loc[a.index]
    for m in ["raw F", "peaks/s", "deconv events/s", "deconv mean", "freeze max", "freeze sum", "freeze max, px/cm 14", "freeze median (variant)", "dart"]:
        t = ss.ttest_rel(a[m], b[m])
        print(f"{gen} (n={len(a)}) {m}: A {a[m].mean():.4f}, B {b[m].mean():.4f}, t({len(a)-1}) = {t.statistic:.2f}, p = {t.pvalue:.3f}")

**Result: not reproduced.** Five of the six activity tests find no difference, but the Drd2+
peak rate is higher in Context A (t(4) = 2.95, p = 0.042), and Drd2-Cre mice do not freeze more in
Context A (p 0.60 to 0.63 under both rules and with a fixed 14 px/cm scale). Drd1a-Cre mice
do freeze more in Context A (p = 0.011). The deconvolved measures and raw fluorescence show no
difference for either type (p 0.29 to 0.81). As in the discrimination sessions, the freezing
frames are those whose pose repeats the previous one within 0.5 px (at most 0.1% of frames fall
between 0.5 and 1.6 px), which is why the scale does not change freezing; the median-displacement variant
gives the same pattern (Drd1a-Cre p = 0.017, Drd2-Cre p = 0.33). The activity part already decides
the outcome: the Drd2+ peak-rate test is significant whatever freezing measures.

## Blocked claims: what was checked

- figs5-ephys-mice-discriminate, figs10lm-defensive-trials-greater-responses,
  fig3fg-no-motor-onset-response: the Open Ephys poses are not on the trial clock (data check).
  A fixed offset is supported by both checks in 2 ASt mice only (3016, 3024; 32 units), and only
  to within about 0.8 s (the spread of the quarter lags in 3024; the shock check agrees with the
  port lag within 0.6 s). fig3fg judges activity 0 to 1 s after bout onsets, so an error of that
  size would blur a response and bias the test toward the claim; figs10lm and figs5 also need the
  unpaired mice, which have no events or behavior. The port-entry half of figs5 uses only
  trial-table events and is printed below for the record; it does not decide the claim.
- fig2o-ast-greater-cs-shock-responses and fig2p-ast-more-responsive compare ASt with TS and CeA.
  No unit is labeled TS. For the record, ASt is printed below against the 22 CeA units of the 2
  Open Ephys CeA mice (the paper has 145 CeA neurons from 8 mice); it does not decide the claims.

In [ ]:
def port_measures(d, ons, dur=20):
    tin = [np.clip(np.minimum(d["port_off"], o + dur) - np.maximum(d["port_on"], o), 0, None).sum() for o in ons]
    n = [((d["port_on"] >= o) & (d["port_on"] < o + dur)).sum() for o in ons]
    return np.mean(tin), np.mean(n)
pm = pd.DataFrame([{"mouse": d["subject_id"][:12], **dict(zip(["shock time in port", "shock entries"], port_measures(d, d["shock_on"]))),
                    **dict(zip(["reward time in port", "reward entries"], port_measures(d, d["reward_on"])))} for d in OE])
print(pm.round(2).to_string())
print("means:", pm.drop(columns="mouse").mean().round(2).to_dict())
print("time in port, CS-reward vs CS-shock:", ss.ttest_rel(pm["reward time in port"], pm["shock time in port"]))
print("entries, CS-reward vs CS-shock:", ss.ttest_rel(pm["reward entries"], pm["shock entries"]))
# fig2o-ast-greater-cs-shock-responses and fig2p-ast-more-responsive need TS units, which no file
# labels. ASt against the 22 CeA units of the two Open Ephys CeA mice, for the record only.
for width in [0.1, 0.01]:
    g = R2O[R2O.width == width]
    for col in ["shock phasic", "shock sustained"]:
        a, b = g[g.region == "ASt"][col].dropna(), g[g.region == "CeA"][col].dropna()
        t = ss.ttest_ind(a, b)
        print(f"bin {width*1000:.0f} ms, CS-{col}: ASt {a.mean():.2f} (n {len(a)}), CeA {b.mean():.2f} (n {len(b)}); F(1,{len(a)+len(b)-2}) = {t.statistic**2:.2f}, p = {t.pvalue:.2g}")
    for rd in "ab":
        for col in ["shock phasic", "reward phasic", "shock sustained"]:
            tab = pd.crosstab(g.region, g[f"{col} class {rd}"]).reindex(columns=[-1, 0, 1], fill_value=0)
            tab = tab.loc[:, tab.sum() > 0]
            share = (tab.drop(columns=0, errors="ignore").sum(axis=1) / tab.sum(axis=1)).round(2).to_dict()
            print(f"  bin {width*1000:.0f} ms, reading ({rd}), CS-{col}: classes (inhibited, none, excited) {pd.crosstab(g.region, g[f'{col} class {rd}']).reindex(columns=[-1, 0, 1], fill_value=0).values.tolist()}, "
                  f"share responsive {share}, chi-square p = {ss.chi2_contingency(tab).pvalue:.3g}")


## Summary

| Claim | Outcome |
|---|---|
| fig2-recorded-neurons | not reproduced |
| figs9-sustained-only-cs-shock | partially reproduced |
| figs10-no-reward-relationship | partially reproduced |
| fig2o-region-differs-cs-reward | partially reproduced |
| fig2q-trajectories-diverge | reproduced |
| figs8-single-trial-divergence | partially reproduced |
| fig3h-cs-decoded-paired-only | not reproduced |
| figs13-imaging-mice-discriminate | reproduced |
| fig5j-drd2-cs-shock-responses | partially reproduced |
| fig5h-drd1-no-difference | not reproduced |
| fig5k-drd2-cs-reward-no-difference | not reproduced |
| fig5m-drd2-rich-clusters-cs-shock | not reproduced |
| figs15-no-context-encoding | not reproduced |
| 35 other claims | blocked (see the header) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the ephys claims use the 10 paired ASt mice recorded
with Open Ephys, as the Neuropixels files have no trial events, and the claims that also need
unpaired, TS or CeA neurons are tested on their paired ASt part only; port latency is the tone
duration; the 90% variant of the fig2q PCA is fitted on paired mice only; the imaging claims use
every CNMF-E ROI, as the paper's curation is not in the files; the pixel scale of the poses is
inferred from the chamber size.

What was found about the dataset: the Neuropixels recordings lack trial events and behavior, and
their units carry a probe depth but no region, so the TS units and all unpaired ephys mice cannot
be analysed; the Open Ephys poses are on a clock offset from the spikes and trial events, by a
constant -9.8 to +6.6 s in 4 of the 12 sessions and by a lag that changes within the other 8,
and nothing in the files says so; the
imaging ROI tables are not curated; one Open Ephys and one imaging recording are deposited
twice, and one copy of the imaging recording has port entries 236 s off its video; in the
imaging poses a fifth of the frames repeat the previous pose within 0.5 px, and the paper's
freezing rule counts only those; the imaging files hold 15 imaged CS-reward clips per session (17 to
19 in seven), not the every-other-trial the Methods describe; the RNAscope, optogenetic and unpredicted-US data
the paper says are in the dandiset are not there; the Open Ephys files do not state the group.

In [ ]:
print("runtime minutes", round((time.time() - T0) / 60, 1), "peak memory MB", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024))